<a href="https://colab.research.google.com/github/SRIKRISHNA1407627/BIS_LAB_1BM24CS151/blob/main/week_04_cuckoo_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import scipy.special as special

class EVCuckooSearch:
    def __init__(self, num_candidates, demand, distance_matrix, fixed_costs,
                 num_nests=25, pa=0.25, max_iter=200, beta=1.5):
        """
        Parameters:
        - num_candidates: Total number of potential EV station locations (m)
        - demand: Array of charging demand at each user node (n)
        - distance_matrix: Shape (m, n) matrix of distances from candidate i to user j
        - fixed_costs: Array of setup costs for each candidate station (m)
        - num_nests: Number of host nests (population size)
        - pa: Discovery rate of alien eggs/solutions (fraction of worst nests abandoned)
        - max_iter: Maximum optimization iterations
        - beta: Lévy flight exponent parameter (typically 1.5)
        """
        self.m = num_candidates
        self.n = len(demand)
        self.demand = demand
        self.dist_mat = distance_matrix
        self.fixed_costs = fixed_costs

        self.num_nests = num_nests
        self.pa = pa
        self.max_iter = max_iter
        self.beta = beta

        # Precompute Lévy flight step scaling factor using Gamma functions
        num = special.gamma(1 + beta) * np.sin(np.pi * beta / 2)
        den = special.gamma((1 + beta) / 2) * beta * (2 ** ((beta - 1) / 2))
        self.sigma = (num / den) ** (1 / beta)

    def fitness(self, nest):
        """
        Calculates total cost. Lower score = better fitness.
        Expects a binary vector where 1 = open station, 0 = closed.
        """
        # Penalty for invalid solution (must open at least one station)
        if np.sum(nest) == 0:
            return float('inf')

        # 1. Total Setup / Fixed Costs
        setup_cost = np.sum(nest * self.fixed_costs)

        # Find indices of active stations
        active_indices = np.where(nest == 1)[0]

        # 2. Total User Travel Cost (Distance * Demand for closest active station)
        active_distances = self.dist_mat[active_indices, :]  # Shape: (num_active, n)
        min_distances = np.min(active_distances, axis=0)      # Distance to closest station for each user
        travel_cost = np.sum(min_distances * self.demand)

        return setup_cost + travel_cost

    def continuous_to_discrete(self, nest_continuous):
        """Maps continuous Lévy flight steps to a binary [0, 1] placement vector."""
        # Use a sigmoid mapping function for binarization
        sigmoid = 1 / (1 + np.exp(-nest_continuous))
        return (sigmoid > 0.5).astype(int)

    def levy_flight(self, current_nest, best_nest):
        """Generates a new solution using Lévy flights distribution."""
        # Draw samples from normal distributions to approximate Lévy stable distribution
        u = np.random.normal(0, self.sigma, size=self.m)
        v = np.random.normal(0, 1, size=self.m)
        step = u / (np.abs(v) ** (1 / self.beta))

        # Global random walk step combined with historical best direction
        step_size = 0.01 * step * (current_nest - best_nest)
        new_nest_continuous = current_nest + step_size * np.random.normal(size=self.m)

        # Enforce bounds to prevent explosion
        new_nest_continuous = np.clip(new_nest_continuous, -5, 5)
        return new_nest_continuous

    def optimize(self):
        # Initialize continuous representations of nests randomly between [-3, 3]
        nests_cont = np.random.uniform(-3, 3, (self.num_nests, self.m))

        # Evaluate initial fitness
        fitness_scores = np.zeros(self.num_nests)
        for idx in range(self.num_nests):
            discrete_nest = self.continuous_to_discrete(nests_cont[idx])
            fitness_scores[idx] = self.fitness(discrete_nest)

        # Identify the initial best nest configuration
        best_idx = np.argmin(fitness_scores)
        best_nest_cont = nests_cont[best_idx].copy()
        best_fitness = fitness_scores[best_idx]

        # Main Optimization Loop
        for iteration in range(self.max_iter):
            # Step 1: Generate a new cuckoo layout using Lévy flight
            cuckoo_idx = np.random.randint(0, self.num_nests)
            new_nest_cont = self.levy_flight(nests_cont[cuckoo_idx], best_nest_cont)

            # Evaluate new solution against a random host nest
            new_discrete = self.continuous_to_discrete(new_nest_cont)
            new_fitness = self.fitness(new_discrete)

            random_host_idx = np.random.randint(0, self.num_nests)
            if new_fitness < fitness_scores[random_host_idx]:
                nests_cont[random_host_idx] = new_nest_cont
                fitness_scores[random_host_idx] = new_fitness

            # Step 2: Discovery of alien eggs (Abandon a fraction 'pa' of worst nests)
            # Sort nests from worst to best fitness
            worst_indices = np.argsort(fitness_scores)[-int(self.num_nests * self.pa):]

            for idx in worst_indices:
                # Random walk step to discover a new host region
                r1, r2 = np.random.randint(0, self.num_nests, 2)
                step_size = np.random.rand() * (nests_cont[r1] - nests_cont[r2])
                nests_cont[idx] = nests_cont[idx] + step_size
                nests_cont[idx] = np.clip(nests_cont[idx], -5, 5)

                # Update fitness score for the rebuilt nest
                disc = self.continuous_to_discrete(nests_cont[idx])
                fitness_scores[idx] = self.fitness(disc)

            # Step 3: Keep track of the globally best configuration
            current_best_idx = np.argmin(fitness_scores)
            if fitness_scores[current_best_idx] < best_fitness:
                best_fitness = fitness_scores[current_best_idx]
                best_nest_cont = nests_cont[current_best_idx].copy()

        # Return final optimal discrete solution vector and its calculated operational cost
        return self.continuous_to_discrete(best_nest_cont), best_fitness

# --- Example Usage Sandbox ---
if __name__ == "__main__":
    np.random.seed(42)  # For reproducible results

    # Simulation Parameters
    CANDIDATES = 10     # 10 potential grid sites across the city
    DEMAND_POINTS = 30  # 30 user demand clusters/neighborhoods

    # Mock data generation
    mock_demand = np.random.randint(10, 100, size=DEMAND_POINTS)      # Vehicles needing charges per zone
    mock_distances = np.random.uniform(0.5, 15.0, size=(CANDIDATES, DEMAND_POINTS))  # Distance in km
    mock_fixed_costs = np.random.uniform(50000, 150000, size=CANDIDATES)             # Fixed installation cost per site

    # Initialize & Run Cuckoo Search Solver
    cs_solver = EVCuckooSearch(
        num_candidates=CANDIDATES,
        demand=mock_demand,
        distance_matrix=mock_distances,
        fixed_costs=mock_fixed_costs,
        num_nests=30,
        max_iter=150
    )

    optimal_layout, minimal_cost = cs_solver.optimize()

    print("=== EV Station Optimizer Output ===")
    print(f"Optimal Station Activation Strategy: {optimal_layout}")
    print(f"Active Station Indices: {np.where(optimal_layout == 1)[0]}")
    print(f"Minimized Combined Cost Score: {minimal_cost:,.2f}")


=== EV Station Optimizer Output ===
Optimal Station Activation Strategy: [0 0 0 0 0 1 0 0 0 0]
Active Station Indices: [5]
Minimized Combined Cost Score: 109,686.00
